## 1. Setup and Configuration

In [ ]:
# Import required libraries
from databricks import sql
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import networkx as nx

In [ ]:
# Load environment variables from parent directory
load_dotenv(dotenv_path='../.env')

# Validate credentials
required_vars = ['DATABRICKS_SERVER_HOSTNAME', 'DATABRICKS_HTTP_PATH', 'DATABRICKS_TOKEN']
missing = [var for var in required_vars if not os.getenv(var)]
if missing:
    raise ValueError(f"Missing environment variables: {', '.join(missing)}")

print("✓ Environment configured")

## 2. Analysis Parameters

In [ ]:
# Customer and time period settings
customer_filter = 'cds_8007'  # SAPPORO DRUG JP
start_date = '2024-01-01'
end_date = '2024-03-31'

# Product filters
category_filter = 'Laundry'
target_condition = "jp_sub_brand_alter_lang_name IN ('アリエール', 'ボールド')"  # Target products

# Co-purchase analysis settings
copurchase_granularity = 'jp_brand_alter_lang_name'  # Product level to analyze
min_support = 10  # Minimum number of co-purchases to consider

print(f"✓ Parameters set")
print(f"  Analysis period: {start_date} to {end_date}")
print(f"  Category: {category_filter}")
print(f"  Target: {target_condition}")
print(f"  Co-purchase granularity: {copurchase_granularity}")
print(f"  Minimum support: {min_support}")

## 3. Build and Execute Query

In [ ]:
# Build co-purchase analysis query
query = f"""
WITH base_transactions AS (
    SELECT
        idpos.shopper_key AS shopper_id,
        CAST(idpos.sales_period_group_end_date_part AS DATE) AS purchase_date,
        {copurchase_granularity} AS product_name,
        pos_unit_sales_qty AS unit,
        pos_sales_amt AS value,
        CASE WHEN {target_condition} THEN 1 ELSE 0 END AS is_target_product,
        CONCAT(idpos.shopper_key, '_', CAST(idpos.sales_period_group_end_date_part AS STRING)) AS basket_id
    FROM
        cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
        LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
        LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
    WHERE
        jp_category_name = '{category_filter}'
        AND idpos.data_provider_code_part = '{customer_filter}'
        AND sales_period_group_end_date_part BETWEEN '{start_date}' AND '{end_date}'
        AND shopper.member_ind = 'Y'
        AND {copurchase_granularity} IS NOT NULL
),
target_baskets AS (
    SELECT DISTINCT basket_id
    FROM base_transactions
    WHERE is_target_product = 1
),
basket_products AS (
    SELECT DISTINCT
        t.basket_id,
        t.product_name,
        t.value,
        t.unit
    FROM base_transactions t
    INNER JOIN target_baskets tb ON t.basket_id = tb.basket_id
),
product_pairs AS (
    SELECT
        a.product_name AS product_a,
        b.product_name AS product_b,
        a.basket_id,
        a.value AS value_a,
        b.value AS value_b
    FROM basket_products a
    INNER JOIN basket_products b 
        ON a.basket_id = b.basket_id 
        AND a.product_name < b.product_name
),
copurchase_counts AS (
    SELECT
        product_a,
        product_b,
        COUNT(DISTINCT basket_id) AS copurchase_count,
        SUM(value_a) AS total_value_a,
        SUM(value_b) AS total_value_b
    FROM product_pairs
    GROUP BY product_a, product_b
    HAVING COUNT(DISTINCT basket_id) >= {min_support}
),
product_basket_counts AS (
    SELECT
        product_name,
        COUNT(DISTINCT basket_id) AS basket_count
    FROM basket_products
    GROUP BY product_name
),
total_baskets AS (
    SELECT COUNT(DISTINCT basket_id) AS total_basket_count
    FROM target_baskets
)
SELECT
    cc.product_a,
    cc.product_b,
    cc.copurchase_count,
    pbc_a.basket_count AS product_a_count,
    pbc_b.basket_count AS product_b_count,
    tb.total_basket_count,
    cc.total_value_a,
    cc.total_value_b,
    -- Support: P(A ∩ B)
    (cc.copurchase_count / tb.total_basket_count) AS support,
    -- Confidence: P(B|A) = P(A ∩ B) / P(A)
    (cc.copurchase_count / pbc_a.basket_count) AS confidence_a_to_b,
    -- Confidence: P(A|B) = P(A ∩ B) / P(B)
    (cc.copurchase_count / pbc_b.basket_count) AS confidence_b_to_a,
    -- Lift: P(A ∩ B) / (P(A) * P(B))
    (cc.copurchase_count * tb.total_basket_count) / (pbc_a.basket_count * pbc_b.basket_count) AS lift
FROM copurchase_counts cc
CROSS JOIN total_baskets tb
LEFT JOIN product_basket_counts pbc_a ON cc.product_a = pbc_a.product_name
LEFT JOIN product_basket_counts pbc_b ON cc.product_b = pbc_b.product_name
ORDER BY cc.copurchase_count DESC
"""

print("✓ SQL query constructed")
print(f"  Query length: {len(query)} characters")

In [ ]:
# Execute query
with sql.connect(
    server_hostname=os.getenv("DATABRICKS_SERVER_HOSTNAME"),
    http_path=os.getenv("DATABRICKS_HTTP_PATH"),
    access_token=os.getenv("DATABRICKS_TOKEN")
) as connection:
    with connection.cursor() as cursor:
        cursor.execute(query)
        result = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description]
        df = pd.DataFrame(result, columns=columns)

# Convert data types
numeric_cols = ['copurchase_count', 'product_a_count', 'product_b_count', 'total_basket_count',
                'total_value_a', 'total_value_b', 'support', 'confidence_a_to_b', 
                'confidence_b_to_a', 'lift']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"✓ Query executed successfully")
print(f"  Retrieved {len(df)} product pairs")
print(f"  Total baskets analyzed: {df['total_basket_count'].iloc[0]:,.0f}")
print(f"\nFirst few rows:")
df.head(10)

## 4. Process and Analyze Results

In [ ]:
# Summary statistics
total_pairs = len(df)
total_baskets = df['total_basket_count'].iloc[0] if len(df) > 0 else 0
avg_lift = df['lift'].mean()
strong_associations = len(df[df['lift'] > 1.5])

# Filter for high-lift associations
high_lift = df[df['lift'] > 1.2].sort_values('lift', ascending=False)

print("=" * 60)
print("CO-PURCHASE ANALYSIS SUMMARY")
print("=" * 60)
print(f"\nOverall Metrics:")
print(f"  Total Baskets Analyzed: {total_baskets:,.0f}")
print(f"  Product Pairs Found: {total_pairs:,}")
print(f"  Average Lift: {avg_lift:.2f}")
print(f"  Strong Associations (Lift > 1.5): {strong_associations}")

print(f"\nTop 10 Co-Purchase Pairs by Frequency:")
for idx, row in df.head(10).iterrows():
    print(f"  {row['product_a']:25s} + {row['product_b']:25s}")
    print(f"    Co-purchases: {row['copurchase_count']:5.0f} | Lift: {row['lift']:5.2f} | Support: {row['support']*100:5.2f}%")

print(f"\nTop 10 Co-Purchase Pairs by Lift:")
for idx, row in high_lift.head(10).iterrows():
    print(f"  {row['product_a']:25s} + {row['product_b']:25s}")
    print(f"    Lift: {row['lift']:5.2f} | Co-purchases: {row['copurchase_count']:5.0f} | Confidence: {row['confidence_a_to_b']*100:.1f}%")

## 5. Visualizations

In [ ]:
# Top co-purchase pairs by frequency
top_20 = df.nlargest(20, 'copurchase_count').copy()
top_20['pair_label'] = top_20['product_a'] + ' + ' + top_20['product_b']

fig = px.bar(
    top_20,
    x='copurchase_count',
    y='pair_label',
    orientation='h',
    title='Top 20 Co-Purchase Pairs by Frequency',
    labels={'copurchase_count': 'Number of Baskets', 'pair_label': 'Product Pair'},
    text='copurchase_count',
    color='lift',
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=1.0
)

fig.update_traces(texttemplate='%{text:.0f}', textposition='outside')
fig.update_layout(
    yaxis={'categoryorder':'total ascending'},
    height=700,
    coloraxis_colorbar_title="Lift"
)

fig.show()

In [ ]:
# Scatter plot: Support vs Lift
scatter_df = df.copy()
scatter_df['pair_label'] = scatter_df['product_a'].str[:15] + ' + ' + scatter_df['product_b'].str[:15]

fig = px.scatter(
    scatter_df,
    x='support',
    y='lift',
    size='copurchase_count',
    hover_data=['product_a', 'product_b', 'copurchase_count', 'confidence_a_to_b'],
    title='Co-Purchase Analysis: Support vs Lift',
    labels={
        'support': 'Support (% of Baskets)',
        'lift': 'Lift (Association Strength)',
        'copurchase_count': 'Co-purchase Count'
    },
    color='lift',
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=1.0
)

# Add reference line at Lift = 1 (no association)
fig.add_hline(y=1.0, line_dash="dash", line_color="gray", opacity=0.5, 
              annotation_text="Lift = 1 (No Association)", annotation_position="right")

fig.update_layout(height=600)
fig.update_xaxes(tickformat='.2%')

fig.show()

In [ ]:
# Network graph of product associations (top 30 pairs)
network_data = df.nlargest(30, 'copurchase_count')

# Create network graph
G = nx.Graph()

# Add edges with weights
for idx, row in network_data.iterrows():
    G.add_edge(
        row['product_a'], 
        row['product_b'], 
        weight=row['copurchase_count'],
        lift=row['lift']
    )

# Get layout
pos = nx.spring_layout(G, k=1, iterations=50)

# Create edge traces
edge_traces = []
for edge in G.edges(data=True):
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    
    # Color by lift
    lift_val = edge[2]['lift']
    if lift_val > 1.5:
        color = 'green'
        width = 3
    elif lift_val > 1.0:
        color = 'orange'
        width = 2
    else:
        color = 'gray'
        width = 1
    
    edge_trace = go.Scatter(
        x=[x0, x1, None],
        y=[y0, y1, None],
        mode='lines',
        line=dict(width=width, color=color),
        hoverinfo='none',
        showlegend=False
    )
    edge_traces.append(edge_trace)

# Create node trace
node_x = []
node_y = []
node_text = []
node_size = []

for node in G.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)
    node_text.append(node)
    # Size by degree (number of connections)
    node_size.append(G.degree(node) * 10)

node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode='markers+text',
    text=node_text,
    textposition="top center",
    textfont=dict(size=10),
    marker=dict(
        size=node_size,
        color='lightblue',
        line=dict(width=2, color='darkblue')
    ),
    hoverinfo='text',
    hovertext=node_text
)

# Create figure
fig = go.Figure(data=edge_traces + [node_trace])

fig.update_layout(
    title='Product Co-Purchase Network (Top 30 Pairs)<br><sub>Node size = # connections | Edge color: Green=Strong (Lift>1.5), Orange=Moderate, Gray=Weak</sub>',
    showlegend=False,
    hovermode='closest',
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    height=800
)

fig.show()

In [ ]:
# Lift distribution histogram
fig = px.histogram(
    df,
    x='lift',
    nbins=50,
    title='Distribution of Lift Values',
    labels={'lift': 'Lift', 'count': 'Number of Product Pairs'},
    color_discrete_sequence=['#4A90E2']
)

fig.add_vline(
    x=1.0,
    line_dash="dash",
    line_color="red",
    annotation_text="Lift = 1 (No Association)",
    annotation_position="top"
)

fig.add_vline(
    x=df['lift'].mean(),
    line_dash="dot",
    line_color="green",
    annotation_text=f"Mean = {df['lift'].mean():.2f}",
    annotation_position="top right"
)

fig.update_layout(height=500)
fig.show()

## 6. Data Tables

In [ ]:
# Top associations by lift
print("Top 20 Product Pairs by Lift (Association Strength):")
high_lift.head(20)[[
    'product_a', 'product_b', 'copurchase_count', 'support', 
    'confidence_a_to_b', 'confidence_b_to_a', 'lift'
]]

In [ ]:
# Top associations by frequency
print("\nTop 20 Product Pairs by Co-Purchase Frequency:")
df.head(20)[[
    'product_a', 'product_b', 'copurchase_count', 'support', 
    'confidence_a_to_b', 'confidence_b_to_a', 'lift'
]]

## 7. Data Export

In [ ]:
# Export to Excel (optional)
export_file = f"copurchase_analysis_{category_filter}_{start_date}_to_{end_date}.xlsx"

with pd.ExcelWriter(export_file, engine='openpyxl') as writer:
    # Summary sheet
    summary_df = pd.DataFrame({
        'Metric': [
            'Total Baskets',
            'Product Pairs Found',
            'Average Lift',
            'Strong Associations (Lift > 1.5)'
        ],
        'Value': [
            f"{total_baskets:,.0f}",
            f"{total_pairs:,}",
            f"{avg_lift:.2f}",
            f"{strong_associations}"
        ]
    })
    summary_df.to_excel(writer, sheet_name='Summary', index=False)
    
    # All pairs
    df.to_excel(writer, sheet_name='All_Pairs', index=False)
    
    # High lift associations
    high_lift.to_excel(writer, sheet_name='High_Lift_Pairs', index=False)
    
    # Top by frequency
    df.nlargest(50, 'copurchase_count').to_excel(writer, sheet_name='Top_Frequency', index=False)

print(f"✓ Data exported to: {export_file}")